# Lab 5: Transformer Encoder with BPE Tokenization

**Course:** CS 417 Large Language Models

**Name:** Muhammad Haseeb Ul Haq &nbsp;&nbsp;&nbsp; **Registration No.:** 454512

**Prerequisites:** This lab builds on Labs 3 and 4. You have already built scaled dot-product attention, sinusoidal positional encoding, and head splitting on small hand-made examples. In this lab you rebuild those pieces on real text: AG News articles, tokenized with Byte Pair Encoding (BPE), processed in batches, and with padding handled correctly. You then put everything together into one complete Transformer encoder block.

**Implementation rules:**

- You may use `torch.matmul` (or `@`), `torch.softmax`, `nn.Embedding`, `nn.Linear`, `nn.LayerNorm`, `nn.Dropout`, `nn.Module`, `nn.Sequential`, activation functions, and ordinary tensor operations.
- You may **not** use `nn.MultiheadAttention`, `nn.Transformer`, `nn.TransformerEncoder`, `nn.TransformerEncoderLayer`, or `torch.nn.functional.scaled_dot_product_attention`. You implement attention yourself.
- You may use the instructor-provided `ignore_padding` helper exactly as given.
- You may use an existing library (Hugging Face `tokenizers`) for BPE tokenization.

The goal is **not** to train a model. It is to understand exactly what happens, and what shape every tensor has, inside a Transformer encoder.

In [ ]:
# Required packages: torch, matplotlib, datasets, tokenizers
# (On Google Colab, uncomment the next line if `datasets` or `tokenizers` is missing.)
# !pip install -q datasets tokenizers

import math
import random

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ---------------- Configuration (use these values) ----------------
SEED       = 42
MAX_LENGTH = 32
VOCAB_SIZE = 8000
BATCH_SIZE = 16
D_MODEL    = 64
NUM_HEADS  = 4
D_HEAD     = D_MODEL // NUM_HEADS   # = 16
D_FF       = 256

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("D_HEAD =", D_HEAD)

---
## Task 1 — Dataset, BPE Tokenization and Padding

### 1.1 Load the AG News Dataset

AG News contains short news articles in four categories: **World**, **Sports**, **Business**, and **Sci/Tech**.

To keep runtime short, build three **disjoint** subsets with a fixed random seed (`SEED`):

| Subset | Size | Drawn from |
|---|---|---|
| Training | 4,000 | official *train* split |
| Validation | 1,000 | official *train* split (no overlap with training) |
| Test | 1,000 | official *test* split |

**TODO:**

1. Shuffle the indices of the official train split with `SEED`. Take the first `N_TRAIN` as `train_idx` and the next `N_VAL` as `val_idx`. Take `N_TEST` shuffled indices of the test split as `test_idx`.
2. Build `train_texts`, `train_labels`, `val_texts`, `val_labels`, `test_texts` and `test_labels` as plain Python lists.
3. Display at least **five** training examples. For each one, show `News Text:` and `Class Label:` (both the integer and the category name).

In [ ]:
from datasets import load_dataset

raw = load_dataset("fancyzhx/ag_news")   # also available as "ag_news"
print(raw)

LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

N_TRAIN, N_VAL, N_TEST = 4000, 1000, 1000

rng = random.Random(SEED)

train_pool = list(range(len(raw["train"])))
rng.shuffle(train_pool)
train_idx = train_pool[:N_TRAIN]
val_idx = train_pool[N_TRAIN:N_TRAIN + N_VAL]

test_pool = list(range(len(raw["test"])))
rng.shuffle(test_pool)
test_idx = test_pool[:N_TEST]

train_batch = raw["train"][train_idx]
train_texts, train_labels = train_batch["text"], [int(y) for y in train_batch["label"]]

val_batch = raw["train"][val_idx]
val_texts, val_labels = val_batch["text"], [int(y) for y in val_batch["label"]]

test_batch = raw["test"][test_idx]
test_texts, test_labels = test_batch["text"], [int(y) for y in test_batch["label"]]


# --- test your implementation ---
assert len(train_texts) == N_TRAIN and len(val_texts) == N_VAL and len(test_texts) == N_TEST
assert set(train_idx).isdisjoint(set(val_idx)), "training and validation subsets must not overlap"
for labels in (train_labels, val_labels, test_labels):
    assert all(isinstance(y, int) for y in labels), "labels must be Python integers"
    assert set(labels) <= {0, 1, 2, 3}, "labels must be integers from 0 to 3"
print("label -> category:", LABEL_NAMES)
print("training label counts:", {LABEL_NAMES[k]: train_labels.count(k) for k in range(4)})


In [ ]:
for i in range(5):
    print("News Text:  ", train_texts[i])
    print("Class Label:", train_labels[i], f"({LABEL_NAMES[train_labels[i]]})")
    print()


#### What I Did

I shuffled the official train-split indices once with `SEED`, took the first 4,000 for training and the next 1,000 for validation (so they can never overlap), and separately shuffled the test split for the 1,000 test indices. Indexing a Hugging Face split with a list of indices returns a dict of lists in one call, which is what `train_batch["text"]` and `train_batch["label"]` pull the texts and labels from.

**Analysis Questions**

- **Q1.** How many classes exist in AG News?
- **Q2.** What does each class represent?
- **Q3.** Why is a labelled news dataset like AG News a reasonable source of text for building and testing a Transformer encoder?

#### My Answers

- **Q1.** AG News has 4 classes.

- **Q2.** The four classes are World (international/political news), Sports, Business (markets, companies, economy), and Sci/Tech (science and technology stories).

- **Q3.** It is short, clean, already-labeled real text covering several genuinely different topics, which is exactly what an encoder needs to produce meaningfully different representations to inspect - long enough to have real multi-token structure for BPE and attention to act on, but short enough (one headline-length article) to fit comfortably inside `MAX_LENGTH = 32` without truncating away most of the content.

### 1.2 Apply BPE Tokenization

Train a small **BPE** tokenizer (not a word-level tokenizer) on the **training texts only**, with `VOCAB_SIZE = 8000` and the special tokens `<PAD>` and `<UNK>`.

The Hugging Face `tokenizers` library works well for this:

```python
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>", continuing_subword_prefix="##"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["<PAD>", "<UNK>"],
    continuing_subword_prefix="##",
)
tokenizer.train_from_iterator(train_texts, trainer)
```

Notes:

- The **whitespace pre-tokenizer** stops merges from crossing word boundaries.
- `continuing_subword_prefix="##"` marks every piece that continues a word (for example `invest`, `##ing`). Without it, a subword piece looks exactly like a whole word.
- Avoid **byte-level** BPE here. It never needs `<UNK>`, but this lab requires one.

**TODO:** Train the tokenizer, then record `pad_id` and `unk_id`.

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

tokenizer = Tokenizer(models.BPE(unk_token="<UNK>", continuing_subword_prefix="##"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["<PAD>", "<UNK>"],
    continuing_subword_prefix="##",
)
tokenizer.train_from_iterator(train_texts, trainer)

pad_id = tokenizer.token_to_id("<PAD>")
unk_id = tokenizer.token_to_id("<UNK>")


# --- test your implementation ---
assert tokenizer is not None
assert pad_id is not None and unk_id is not None and pad_id != unk_id
assert tokenizer.get_vocab_size() <= VOCAB_SIZE
print("vocabulary size:", tokenizer.get_vocab_size())
print("<PAD> id:", pad_id, "| <UNK> id:", unk_id)


#### What I Did

I trained a BPE model on the training texts only, with a whitespace pre-tokenizer so merges never cross word boundaries, and `##` marking any piece that continues a word rather than starting one. `<PAD>` and `<UNK>` are reserved as special tokens, so their IDs are fixed and never get merged away.

### 1.3 Inspect BPE Output

**TODO:** Implement `show_bpe(text)`. It prints the original text, its BPE tokens, and its token IDs. Run it on three sentences from the dataset.

Example (your tokens and IDs will be different):

```text
Original:  "Technology companies are investing heavily in artificial intelligence."
BPE Tokens: ['Technology', 'companies', 'are', 'invest', '##ing', 'heavily', 'in', 'artificial', 'intelligence', '.']
Token IDs:  [421, 812, 44, 935, 67, 1508, 31, 972, 681, 17]
```

In [ ]:
def show_bpe(text):
    enc = tokenizer.encode(text)
    print("Original:  ", repr(text))
    print("BPE Tokens:", enc.tokens)
    print("Token IDs: ", enc.ids)


for text in train_texts[:3]:
    show_bpe(text)
    print()


#### What I Did

`show_bpe` just encodes the text and prints the three things that matter: the raw string, the BPE token pieces, and their numeric IDs - so I can see exactly what the tokenizer did to real sentences before anything gets padded or embedded.

### 1.4 Examine Subword Splitting

**TODO:** Implement `find_split_words(text)`. It returns a list of `(word, pieces)` pairs, one for every word that BPE split into **more than one** token.

*Hint:* `encoding.word_ids` gives, for each token, the index of the word it came from (`None` for padding or special tokens). `encoding.offsets` gives each token's character span in the original text.

In [ ]:
def find_split_words(text):
    enc = tokenizer.encode(text)
    groups = {}
    for pos, word_id in enumerate(enc.word_ids):
        if word_id is None:
            continue
        groups.setdefault(word_id, []).append(pos)

    results = []
    for word_id, positions in groups.items():
        if len(positions) > 1:
            start = enc.offsets[positions[0]][0]
            end = enc.offsets[positions[-1]][1]
            word = text[start:end]
            pieces = [enc.tokens[p] for p in positions]
            results.append((word, pieces))
    return results


# --- test your implementation ---
found = []
for text in train_texts[:500]:
    found += find_split_words(text)
    if len(found) >= 5:
        break
assert len(found) > 0, "expected at least one word split into several BPE tokens"
for word, pieces in found[:5]:
    assert len(pieces) > 1
    print(f"Original word: {word!r:25s} -> BPE representation: {pieces}")


#### What I Did

I grouped token positions by `enc.word_ids` (skipping `None`, which marks special/padding tokens), kept only the words that ended up with more than one token, and used `enc.offsets` to slice the original word back out of the text so I can print it next to its BPE pieces.

**Analysis Questions**

- **Q4.** Why does BPE sometimes divide a word into multiple tokens?
- **Q5.** What advantage does BPE provide compared with a purely word-level tokenizer?
- **Q6.** How can BPE handle a rare word that was not present as a complete word during training?

#### My Answers

- **Q4.** BPE only keeps a fixed-size vocabulary of the most frequent character/subword merges. A word that wasn't frequent enough to earn its own whole-word slot during training simply doesn't have one, so the tokenizer falls back to representing it as a sequence of smaller pieces that *are* in the vocabulary.

- **Q5.** A word-level tokenizer needs one vocabulary slot per distinct word and maps anything it has never seen to a single `<UNK>`, throwing that word's information away completely. BPE instead reuses a much smaller set of subword pieces to spell out rare or unseen words, so the model still gets partial, meaningful information (e.g. the stem of a word plus a suffix) instead of one opaque "unknown" symbol.

- **Q6.** Even if a rare word never appeared as a whole word during training, its individual characters and common substrings almost certainly did appear in other words, so BPE can still spell it out piece by piece from vocabulary entries it already has - worst case, falling all the way back to individual characters, which are guaranteed to be in the vocabulary since BPE starts from characters.

### 1.5 Padding and Truncation

Every article must have **exactly** `MAX_LENGTH = 32` positions. Shorter articles are padded with `<PAD>`, and longer articles are truncated.

With `tokenizers`, use `tokenizer.enable_truncation(max_length=MAX_LENGTH)` and `tokenizer.enable_padding(pad_id=pad_id, pad_token="<PAD>", length=MAX_LENGTH)`.

Without `length=MAX_LENGTH`, `enable_padding` only pads each batch to its longest sequence, not to 32.

After padding, build the Boolean **valid-token indicator**:

```python
valid_tokens = token_ids != pad_id
```

```text
Tokens:       [AI, changes, business, <PAD>, <PAD>]
Valid tokens: [True, True,  True,     False, False]
```

`True` marks article content and `False` marks padding. Keep this indicator alongside every batch, because attention uses it. Every article must contain at least one real token. Otherwise every key in that row is masked and softmax returns `NaN`.

**Note:** Use this exactly as given. It stops tokens from attending to **padding** positions: softmax turns `-inf` scores into zero probability. It does not restrict attention between real tokens.

**TODO:**

1. Enable truncation and padding as described above.
2. Implement `encode_texts(texts)`. It returns a `torch.long` tensor of shape `[len(texts), MAX_LENGTH]`.
3. Display one sample: Original Text, BPE Tokens, Token IDs (before padding), and Padded/Truncated Token IDs.

In [ ]:
# ---------------- Instructor-provided helper (do not modify) ----------------
def ignore_padding(scores, valid_tokens):
    # scores: [B, L, L] for single-head attention
    #      or [B, H, L, L] for multi-head attention
    if scores.ndim == 3:
        valid_keys = valid_tokens[:, None, :]
    elif scores.ndim == 4:
        valid_keys = valid_tokens[:, None, None, :]
    else:
        raise ValueError("Expected 3D or 4D attention scores")
    return scores.masked_fill(~valid_keys, float("-inf"))

In [ ]:
tokenizer.enable_truncation(max_length=MAX_LENGTH)
tokenizer.enable_padding(pad_id=pad_id, pad_token="<PAD>", length=MAX_LENGTH)


def encode_texts(texts):
    encodings = tokenizer.encode_batch(texts)
    ids = [enc.ids for enc in encodings]
    return torch.tensor(ids, dtype=torch.long)


train_ids = encode_texts(train_texts)
val_ids = encode_texts(val_texts)
test_ids = encode_texts(test_texts)

train_valid = train_ids != pad_id
val_valid = val_ids != pad_id
test_valid = test_ids != pad_id

# The first batch: used throughout Tasks 2-7
token_ids = train_ids[:BATCH_SIZE]
valid_tokens = train_valid[:BATCH_SIZE]


# --- test your implementation ---
assert train_ids.dtype == torch.long
assert train_ids.shape == (N_TRAIN, MAX_LENGTH)
assert val_ids.shape == (N_VAL, MAX_LENGTH) and test_ids.shape == (N_TEST, MAX_LENGTH)
for v in (train_valid, val_valid, test_valid):
    assert (v.sum(dim=1) > 0).all(), "every article must contain at least one real token"
assert token_ids.shape == (BATCH_SIZE, MAX_LENGTH)
assert valid_tokens.dtype == torch.bool and valid_tokens.shape == token_ids.shape
print("Token IDs shape:      ", token_ids.shape)
print("Valid-token shape:    ", valid_tokens.shape)
print("real tokens per article in this batch:", valid_tokens.sum(dim=1).tolist())


#### What I Did

I enabled truncation and padding to exactly `MAX_LENGTH`, then `encode_texts` runs `tokenizer.encode_batch` and stacks the resulting IDs into one `long` tensor. `valid_tokens = token_ids != pad_id` marks every real-content position `True` and every padding position `False`, which is the indicator attention needs to ignore padding.

In [ ]:
sample_idx = 0
sample_text = train_texts[sample_idx]

tokenizer.no_padding()
tokenizer.no_truncation()
enc_raw = tokenizer.encode(sample_text)
tokenizer.enable_truncation(max_length=MAX_LENGTH)
tokenizer.enable_padding(pad_id=pad_id, pad_token="<PAD>", length=MAX_LENGTH)

print("Original Text:             ", sample_text)
print("BPE Tokens:                ", enc_raw.tokens)
print("Token IDs (before padding):", enc_raw.ids)
print("Padded/Truncated Token IDs:", train_ids[sample_idx].tolist())
print("valid_tokens row:          ", train_valid[sample_idx].tolist())


#### What I Did

To see the IDs *before* padding I temporarily switch padding/truncation off, encode the sample, then switch both back on exactly as they were - so the rest of the notebook keeps using the padded tokenizer.

**Analysis Question**

- **Q7.** Why do sequences in the same batch usually need the same length?

#### My Answers

- **Q7.** PyTorch tensors are rectangular: every row in a batch has to have the same number of columns so they can sit in one tensor and go through the same matrix operations together. If articles kept their natural, different lengths, they couldn't be stacked into a single `[B, L]` tensor at all, so every article is padded or truncated to the same fixed length first, and the `valid_tokens` indicator is what keeps the padding from quietly contaminating the actual computation.

---
## Task 2 — Token Embeddings and Positional Encoding

### 2.1 Create Token Embeddings

**TODO:** Create an embedding layer with one 64-dimensional vector per vocabulary entry. Use `padding_idx=pad_id` so the `<PAD>` vector stays at zero. Pass the batch `token_ids` through it.

```text
Token IDs [16, 32]  --Embedding-->  [16, 32, 64]
```

In [ ]:
torch.manual_seed(SEED)

embedding = nn.Embedding(num_embeddings=tokenizer.get_vocab_size(), embedding_dim=D_MODEL, padding_idx=pad_id)
embeddings = embedding(token_ids)


# --- test your implementation ---
print("Token IDs shape:", token_ids.shape)
print("Embedding shape:", embeddings.shape)
assert embeddings.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL)
assert embedding.num_embeddings == tokenizer.get_vocab_size()
assert torch.all(embedding.weight[pad_id] == 0), "the <PAD> embedding should be all zeros"


#### What I Did

`nn.Embedding` is just a lookup table with one learnable 64-dim row per vocabulary ID. Passing `padding_idx=pad_id` tells PyTorch to keep that one row permanently at zero and never update it, so padding positions contribute nothing extra to the embedding.

**Shape Interpretation.** For `[16, 32, 64]`, write in your own words what each dimension represents:

- 16 → the **batch size**: how many articles are processed together in this batch.
- 32 → the **sequence length**: the fixed number of token positions per article (`MAX_LENGTH`), padded or truncated to match.
- 64 → **d_model**: the size of the embedding vector representing each individual token.

### 2.2 Add Positional Encoding

Self-attention does not inherently know whether a token occurs at position 1, 5 or 20. Implement **sinusoidal positional encoding** (as in Lab 4):

$$PE(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad PE(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

and **add** it to the token embeddings. Do **not** concatenate:

$$X = \text{Embedding} + \text{PositionalEncoding}$$

```text
Token embeddings [16, 32, 64]  +  Positional encoding [32, 64]  ->  X [16, 32, 64]
```

**TODO:** Implement `positional_encoding(seq_len, d_model)`. Then compute `X` using broadcasting.

In [ ]:
def positional_encoding(seq_len, d_model):
    position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
    dim_index = torch.arange(d_model, dtype=torch.float32).unsqueeze(0)
    angle_rates = 1.0 / (10000 ** (2 * (dim_index // 2) / d_model))
    angles = position * angle_rates

    PE = torch.zeros(seq_len, d_model)
    PE[:, 0::2] = torch.sin(angles[:, 0::2])
    PE[:, 1::2] = torch.cos(angles[:, 1::2])
    return PE


PE = positional_encoding(MAX_LENGTH, D_MODEL)

X = embeddings + PE.unsqueeze(0)


# --- test your implementation ---
print("Token embeddings shape:   ", embeddings.shape)
print("Positional encoding shape:", PE.shape)
print("After addition (X) shape: ", X.shape)
assert PE.shape == (MAX_LENGTH, D_MODEL)
assert torch.allclose(PE[0], torch.tensor([0.0, 1.0] * (D_MODEL // 2)), atol=1e-6)
assert not torch.allclose(PE[1], PE[2])
assert X.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL), "PE must be added, not concatenated"
assert torch.allclose(X - embeddings, PE.unsqueeze(0).expand_as(X), atol=1e-6)


#### What I Did

Same sinusoidal formula as Lab 4: even columns get `sin`, odd columns get `cos`, each pair sharing one wavelength. `PE` has shape `[32, 64]` - one fixed vector per position, shared by every article - and adding it to `embeddings` broadcasts it across the batch dimension so every article's tokens pick up the same position-dependent signal.

**Analysis Questions**

- **Q8.** Why do Transformers require positional information?
- **Q9.** Why does adding positional encoding not change `d_model` from 64?
- **Q10.** What would happen if positional encoding were concatenated instead of added?

#### My Answers

- **Q8.** Self-attention computes everything from dot products between embeddings, and nothing in that computation looks at where a token physically sits in the sequence - so without an extra signal, a Transformer would treat a sentence and any reordering of its words identically. Positional encoding is what supplies that missing "where" information.

- **Q9.** Positional encoding is **added** to the embedding, not appended as new columns, so it reuses the same 64 dimensions the embedding already has instead of creating extra ones. The shape stays `[..., 64]` throughout; only the *values* inside those 64 numbers change.

- **Q10.** Concatenating would increase `d_model` (e.g. to 64 + 64 = 128 if the positional vector were also 64-wide), which would ripple through every downstream shape - `W_Q`/`W_K`/`W_V`, the attention score dimension, `d_head`, the FFN's input width - all of it would need to change to match. Addition avoids all of that by keeping word identity and position packed into the same fixed-size vector.

---
## Task 3 — Single-Head Self-Attention from Scratch

Build **single-head** attention first, before moving to multiple heads, and print each intermediate shape as you go.

### 3.1 Generate Query, Key and Value

$$Q = XW_Q, \qquad K = XW_K, \qquad V = XW_V$$

**TODO:**

1. Create `W_Q`, `W_K` and `W_V` as `nn.Linear(D_MODEL, D_MODEL, bias=False)`.
2. Compute `Q`, `K` and `V` from `X`.
3. Choose one real token from one sentence. Display the first 8–10 dimensions of its input vector and its Query, Key and Value vectors.

In [ ]:
torch.manual_seed(SEED)

W_Q = nn.Linear(D_MODEL, D_MODEL, bias=False)
W_K = nn.Linear(D_MODEL, D_MODEL, bias=False)
W_V = nn.Linear(D_MODEL, D_MODEL, bias=False)

Q = W_Q(X)
K = W_K(X)
V = W_V(X)


# --- test your implementation ---
for name, t in [("X", X), ("Q", Q), ("K", K), ("V", V)]:
    print(f"{name}: {tuple(t.shape)}")
    assert t.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL)
assert W_Q.bias is None and W_K.bias is None and W_V.bias is None
assert not torch.allclose(Q, K), "Q and K should differ: they use different weights"


#### What I Did

`W_Q`, `W_K`, `W_V` are three separate, bias-free linear layers, so each produces a different linear mix of the same input `X`. Q, K, and V end up numerically different purely because the three weight matrices are different, not because the input changed.

In [ ]:
EX = 0
real_positions = torch.nonzero(valid_tokens[EX]).flatten()
pos = int(real_positions[min(2, len(real_positions) - 1)])

token_str = tokenizer.encode(train_texts[EX]).tokens[pos]
print(f"Token at position {pos} in article {EX}: {token_str!r}\n")

print("X[EX, pos]:", X[EX, pos, :10])
print("Q[EX, pos]:", Q[EX, pos, :10])
print("K[EX, pos]:", K[EX, pos, :10])
print("V[EX, pos]:", V[EX, pos, :10])


#### What I Did

I picked the third real (non-padding) token of the first article and printed the first 10 dimensions of its input vector alongside its Query, Key, and Value vectors, so the three projections can be compared side by side for one concrete token.

**Analysis Questions**

- **Q11.** Q, K and V are all generated from the same input X. Why do they have different numerical values?
- **Q12.** What is the conceptual role of the Query, the Key, and the Value?

#### My Answers

- **Q11.** `W_Q`, `W_K`, and `W_V` are three independently initialized weight matrices, so `X @ W_Q`, `X @ W_K`, and `X @ W_V` are three different linear transformations of the same input - different matrices multiplying the same vector naturally give different results, even though all three start from identical `X`.

- **Q12.** The **Query** represents what a token is currently "looking for" in the rest of the sequence. The **Key** represents what a token "offers" or advertises about itself, which gets compared against every other token's Query. The **Value** is the actual content that gets mixed into the output once the Query-Key comparison has decided how much attention to pay to that token.

### 3.2 Compute Raw Attention Scores

$$\text{Scores} = QK^{T}$$

```text
Q [16, 32, 64]   K^T [16, 64, 32]   ->   QK^T [16, 32, 32]
```

**TODO:** Compute `K_T` and `scores` with `torch.matmul`, and print all three shapes.

In [ ]:
K_T = K.transpose(-2, -1)
scores = torch.matmul(Q, K_T)


# --- test your implementation ---
print("Q shape:     ", Q.shape)
print("K^T shape:   ", K_T.shape)
print("Scores shape:", scores.shape)
assert K_T.shape == (BATCH_SIZE, D_MODEL, MAX_LENGTH)
assert scores.shape == (BATCH_SIZE, MAX_LENGTH, MAX_LENGTH)
assert torch.allclose(scores, torch.einsum("bid,bjd->bij", Q, K), atol=1e-4)
print("Attention scores for one sentence:", scores[0].shape)


#### What I Did

`K.transpose(-2, -1)` swaps the last two dimensions so `K^T` is `[B, 64, 32]`, letting `Q @ K^T` contract over the 64 feature dimensions and leave a `[B, 32, 32]` matrix of every query-token-vs-key-token dot product, per article.

**Analysis Questions**

- **Q13.** Why is the attention matrix 32 × 32 rather than 32 × 64?
- **Q14.** What does one row of this matrix represent?
- **Q15.** What does one column represent?

#### My Answers

- **Q13.** The attention matrix compares every query token against every key token, so its shape is `n x n` (32 x 32 here) - the 64 feature dimensions are summed away inside each dot product, not preserved in the output.

- **Q14.** Row *i* holds how strongly token *i*'s Query matches the Key of every token in the article - it's token *i*'s full set of comparison scores against everyone else.

- **Q15.** Column *j* holds how strongly every token's Query matches token *j*'s Key - it's how much attention token *j* receives from every query in the article.

### 3.3 Scale the Attention Scores

$$\text{ScaledScores} = \frac{QK^{T}}{\sqrt{d_k}}, \qquad d_k = 64 \text{ for this single-head example}$$

**TODO:** Compute `scaled_scores`.

In [ ]:
scaled_scores = scores / math.sqrt(D_MODEL)


# --- test your implementation ---
assert torch.allclose(scaled_scores, scores / 8.0, atol=1e-5)
print("scores std:       ", scores.std().item())
print("scaled scores std:", scaled_scores.std().item())


#### What I Did

I divided every score by `sqrt(D_MODEL) = sqrt(64) = 8`, which matches the single-head case here since `d_k = D_MODEL`. The standard deviation printed for `scaled_scores` is noticeably smaller than for the raw `scores`, exactly as the scaling is meant to do.

**Analysis Question**

- **Q16.** Why do we scale the dot-product scores before applying softmax?

#### My Answers

- **Q16.** A dot product between two `d_k`-dimensional vectors tends to grow in magnitude as `d_k` grows, simply because there are more terms being summed. Without scaling, those large scores push softmax into a very peaked, saturated regime where almost all the gradient is near zero, making learning slow or unstable - dividing by `sqrt(d_k)` keeps the scores in a sensible range regardless of dimension.

### 3.4 Apply Softmax

First hide the padding keys with the instructor-provided helper. Then turn each row into probabilities:

$$A = \text{Softmax}(\text{ScaledScores})$$

**TODO:**

1. `masked_scores = ignore_padding(scaled_scores, valid_tokens)`
2. `attention_weights = torch.softmax(masked_scores, dim=-1)`
3. For one row, print `attention_weights[0, some_token, :].sum()` and the weights given to padding positions, `attention_weights[0, some_token, ~valid_tokens[0]]`.

In [ ]:
masked_scores = ignore_padding(scaled_scores, valid_tokens)
attention_weights = torch.softmax(masked_scores, dim=-1)

some_token = pos
print("Row sum for one query token:", attention_weights[0, some_token, :].sum().item())
print("Weights given to padding positions:", attention_weights[0, some_token, ~valid_tokens[0]])


# --- test your implementation ---
print("Attention weights shape:", attention_weights.shape)
assert attention_weights.shape == (BATCH_SIZE, MAX_LENGTH, MAX_LENGTH)
assert torch.allclose(attention_weights.sum(dim=-1), torch.ones(BATCH_SIZE, MAX_LENGTH), atol=1e-5), \
    "every attention row must sum to 1"
pad_mass = (attention_weights * (~valid_tokens)[:, None, :]).abs().max().item()
assert pad_mass < 1e-6, "padding key positions must receive (approximately) zero attention"
print("max attention given to any padding key:", pad_mass)


#### What I Did

`ignore_padding` sets the score of every padding key to `-inf` before softmax, so `exp(-inf) = 0` and those positions get exactly zero probability no matter what their raw score was. The printed padding weights are all 0, and every row still sums to 1 over just the real tokens.

**Analysis Question**

- **Q17.** Why does every row approximately sum to 1?

#### My Answers

- **Q17.** Softmax by construction divides every value in a row by the sum of all of them, so the row's outputs always add up to 1 - that's what makes it a valid probability distribution. It's only "approximately" 1 in printed output because of ordinary floating-point rounding, not because anything is conceptually off.

### 3.5 Generate Contextualized Representations

$$\text{Context} = AV \qquad A\,[16, 32, 32] \;\times\; V\,[16, 32, 64] \;\rightarrow\; [16, 32, 64]$$

**TODO:** Compute `context` manually and print the three shapes.

In [ ]:
context = torch.matmul(attention_weights, V)


# --- test your implementation ---
print("Attention matrix shape:", attention_weights.shape)
print("Value shape:           ", V.shape)
print("Context output shape:  ", context.shape)
assert context.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL)
assert torch.allclose(context, torch.einsum("bij,bjd->bid", attention_weights, V), atol=1e-5)


#### What I Did

`attention_weights @ V` turns each row of weights into a weighted average of all the Value vectors, so every output position becomes one `[64]`-dim mixture built from the Values of whichever real tokens that row attended to.

**Analysis Question**

- **Q18.** Why is the output representation for one token now influenced by the representations of other tokens?

#### My Answers

- **Q18.** Because `context[i]` is a weighted sum over *every* token's Value vector, not just token `i`'s own - the weights come from how much token `i`'s Query matched every Key, so other tokens' content gets blended into token `i`'s output in proportion to how relevant they were judged to be. That mixing across tokens is exactly what makes the representation "contextualized" instead of just a copy of the original embedding.

---
## Task 4 — Visualize and Interpret Self-Attention

### 4.1 Visualize Self-Attention

Pick one short article from the batch. Plot its attention matrix as a heatmap over the **first 10–15 real tokens only**, excluding padding. Label the rows and columns with the BPE tokens.

Rows will not sum to 1 in this plot, because attention to the tokens you leave out is not shown.

**TODO:** Implement `plot_attention(weights, tokens, title)`. Use `plt.imshow`, put token labels on both axes (rotate the x labels), label the axes "Key"/"Query", and add a colorbar and a title.

In [ ]:
def plot_attention(weights, tokens, title):
    weights_np = weights.numpy() if torch.is_tensor(weights) else weights
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(weights_np, cmap="viridis")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticklabels(tokens)
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="attention weight")
    plt.tight_layout()
    plt.show()


EX = 0                                                  # article index within the batch
n_show = min(15, int(valid_tokens[EX].sum()))           # real tokens only
tokens_ex = tokenizer.encode(train_texts[EX]).tokens[:n_show]
print(f"showing {n_show} real tokens:", tokens_ex)

plot_attention(attention_weights[EX, :n_show, :n_show].detach(), tokens_ex,
               "Single-head self-attention (untrained)")


#### What I Did

`plot_attention` draws the sliced `[n, n]` weight matrix as a heatmap with the actual BPE token strings labeling both axes (Query rows, Key columns), plus a colorbar so the shading has a numeric scale. I only show the first 15 real tokens of one article, excluding padding, so the labels stay readable.

### 4.2 Interpret One Row

**TODO:** Implement `describe_row(weights, tokens, row)`. For the selected query token, report:

```text
Selected token:
Token receiving highest attention:   Attention value:
Token receiving lowest attention:    Attention value:
```

In [ ]:
def describe_row(weights, tokens, row):
    row_weights = weights[row]
    max_idx = int(torch.argmax(row_weights))
    min_idx = int(torch.argmin(row_weights))
    print("Selected token:", tokens[row])
    print(f"Token receiving highest attention: {tokens[max_idx]!r}   Attention value: {row_weights[max_idx]:.4f}")
    print(f"Token receiving lowest attention:  {tokens[min_idx]!r}   Attention value: {row_weights[min_idx]:.4f}")


describe_row(attention_weights[EX, :n_show, :n_show].detach(), tokens_ex, row=2)


#### What I Did

`describe_row` just reads off one query token's row, finds the index of its largest and smallest weight with `argmax`/`argmin`, and prints both the token strings and their attention values, so a single row of the heatmap can be inspected as exact numbers instead of just color.

**Analysis Question**

- **Q19.** The model is untrained. Should you interpret high attention as a meaningful linguistic relationship? Why or why not? *Hint: think about what determines the scores when the projection weights are random, including the role of positional encoding.*

#### My Answers

- **Q19.** No - with random, untrained `W_Q`/`W_K`/`W_V`, any pattern in the heatmap is a product of those random projections plus the sinusoidal positional encoding, not anything the model has learned about language. Because positional encoding gives nearby positions more similar vectors than far-apart ones, even an untrained encoder can show a mild bias toward attending to nearby tokens - that's a geometric artifact of the encoding scheme, not evidence the model understands grammar or meaning. A high weight here should be read as "these two token-plus-position vectors happened to have a large dot product under this particular random initialization," nothing more.

---
## Task 5 — Multi-Head Self-Attention

### 5.1 Split Q, K and V into Multiple Heads

With `D_MODEL = 64` and `NUM_HEADS = 4`, each head works with $d_{head} = 64 / 4 = 16$ dimensions:

```text
[B, 32, 64]  --reshape-->  [B, 32, 4, 16]  --transpose(1, 2)-->  [B, 4, 32, 16]
```

**TODO:**

1. Implement `split_heads(x, num_heads)`: `[B, L, D]` → `[B, H, L, D/H]`. Raise a `ValueError` if `D` is not divisible by `num_heads`. Read `B` and `L` from `x.shape` rather than hard-coding them, because the final batch can be smaller than 16.
2. Implement `combine_heads(x)`: the exact inverse, `[B, H, L, d_head]` → `[B, L, H·d_head]`.
   *Hint:* after `transpose`, a tensor is no longer contiguous in memory. Use `.reshape(...)` or `.contiguous().view(...)`, because a plain `.view(...)` will fail.
3. Split `Q`, `K` and `V`. Print the shape after every step.

In [ ]:
def split_heads(x, num_heads):
    B, L, D = x.shape
    if D % num_heads != 0:
        raise ValueError(f"d_model ({D}) must be evenly divisible by num_heads ({num_heads}).")
    d_head = D // num_heads
    x = x.reshape(B, L, num_heads, d_head)
    return x.transpose(1, 2)


def combine_heads(x):
    B, H, L, d_head = x.shape
    x = x.transpose(1, 2)
    return x.reshape(B, L, H * d_head)


Q_h = split_heads(Q, NUM_HEADS)
K_h = split_heads(K, NUM_HEADS)
V_h = split_heads(V, NUM_HEADS)
print("Q after split_heads:", Q_h.shape)
print("K after split_heads:", K_h.shape)
print("V after split_heads:", V_h.shape)


# --- test your implementation ---
for name, t in [("Q_h", Q_h), ("K_h", K_h), ("V_h", V_h)]:
    print(f"{name}: {tuple(t.shape)}")
    assert t.shape == (BATCH_SIZE, NUM_HEADS, MAX_LENGTH, D_HEAD)
assert torch.allclose(combine_heads(Q_h), Q), "combine_heads must exactly undo split_heads"
assert torch.allclose(Q_h[:, 1], Q[:, :, D_HEAD:2 * D_HEAD]), "head 1 should hold columns 16-31"
assert split_heads(Q[:5], NUM_HEADS).shape == (5, NUM_HEADS, MAX_LENGTH, D_HEAD), \
    "split_heads must work for any batch size"
try:
    split_heads(Q, 5)
    raised = False
except ValueError:
    raised = True
assert raised, "split_heads should reject a num_heads that does not divide d_model"


#### What I Did

`split_heads` reshapes `[B, L, D]` into `[B, L, H, d_head]` (cutting the 64 columns into 4 contiguous groups of 16) and then transposes to `[B, H, L, d_head]` so each head looks like its own independent batch. `combine_heads` reverses that exact transpose-then-reshape. Both read `B` and `L` straight from the input shape, so they work for the final, smaller batch too, and `split_heads` raises `ValueError` up front if `num_heads` doesn't evenly divide `d_model`.

**Complete the Shape Trace** (repeat for K and V)

| | Q | K | V |
|---|---|---|---|
| Original | `[16, 32, 64]` | `[16, 32, 64]` | `[16, 32, 64]` |
| After reshape | `[16, 32, 4, 16]` | `[16, 32, 4, 16]` | `[16, 32, 4, 16]` |
| After transpose | `[16, 4, 32, 16]` | `[16, 4, 32, 16]` | `[16, 4, 32, 16]` |

**Analysis Question**

- **Q20.** Why does each head use 16 projected dimensions? Explain why these projected dimensions can depend on all 64 input features.

#### My Answer

Each head uses 16 dimensions because `d_head = d_model / num_heads = 64 / 4 = 16` - splitting is just cutting the 64 projected columns into 4 equal, contiguous blocks. Even though each head only *keeps* 16 of those columns, every one of those 16 values was computed as `X @ W_Q` (a full matrix multiply against all 64 input features) before any splitting happened - `W_Q` is a `[64, 64]` matrix, so column 5 of `Q`, for instance, is already a weighted combination of all 64 input dimensions. Splitting only decides which already-mixed columns go to which head; it never throws away information from the other 48 input features.

### 5.2 Calculate Attention Independently for Every Head

$$\text{Attention}(Q, K, V) = \text{Softmax}\!\left(\frac{QK^{T}}{\sqrt{d_{head}}}\right)V$$

```text
Q_h [16, 4, 32, 16]   K_h^T [16, 4, 16, 32]   ->   scores [16, 4, 32, 32]   ->   output [16, 4, 32, 16]
```

Order: **scale → ignore padding (provided helper) → softmax → multiply with V**. Every head uses the same `valid_tokens` indicator, and each real token can still attend to every real token in its article.

**TODO:** Compute every intermediate below and print its shape.

In [ ]:
mh_scores = Q_h @ K_h.transpose(-2, -1)                      # Q_h @ K_h^T
mh_scaled = mh_scores / math.sqrt(D_HEAD)                     # / sqrt(D_HEAD)
mh_masked = ignore_padding(mh_scaled, valid_tokens)            # ignore_padding(...)
mh_weights = torch.softmax(mh_masked, dim=-1)                  # softmax over the last dimension
mh_context = mh_weights @ V_h                                  # mh_weights @ V_h


# --- test your implementation ---
print("scores:   ", tuple(mh_scores.shape))
print("weights:  ", tuple(mh_weights.shape))
print("per-head output:", tuple(mh_context.shape))
assert mh_scores.shape == (BATCH_SIZE, NUM_HEADS, MAX_LENGTH, MAX_LENGTH)
assert torch.allclose(mh_scaled, mh_scores / math.sqrt(D_HEAD), atol=1e-5)
assert torch.allclose(mh_weights.sum(dim=-1), torch.ones(BATCH_SIZE, NUM_HEADS, MAX_LENGTH), atol=1e-5)
assert (mh_weights * (~valid_tokens)[:, None, None, :]).abs().max().item() < 1e-6
assert mh_context.shape == (BATCH_SIZE, NUM_HEADS, MAX_LENGTH, D_HEAD)


#### What I Did

This is exactly the same five steps as single-head attention in Task 3 - score, scale, mask padding, softmax, multiply by V - just run on `[B, H, L, d_head]` tensors instead of `[B, L, D]`. `ignore_padding` and `torch.softmax` don't need to change at all, since they only ever touch the last one or two dimensions and treat the head axis as another batch dimension automatically.

**Shape Interpretation.** For the attention tensor `[16, 4, 32, 32]`, explain each dimension:

- 16 = **batch size** - number of articles in this batch.
- 4 = **number of heads** - each head computes its own, independent attention pattern.
- 32 = **query positions** - one row per token asking "what should I attend to?"
- 32 = **key positions** - one column per token being attended to, for that query.

### 5.3 Concatenate Attention Heads

```text
[16, 4, 32, 16]  --transpose-->  [16, 32, 4, 16]  --concatenate-->  [16, 32, 64]  --W_O-->  [16, 32, 64]
```

The Q, K and V projections have no bias, while `W_O = nn.Linear(D_MODEL, D_MODEL)` keeps PyTorch's default bias. This choice does not affect any shape.

**TODO:** Create `W_O`. Combine the heads with `combine_heads` and apply the output projection.

In [ ]:
torch.manual_seed(SEED)

W_O = nn.Linear(D_MODEL, D_MODEL)
concat_heads = combine_heads(mh_context)
mha_out = W_O(concat_heads)


# --- test your implementation ---
print("Concatenated heads:", tuple(concat_heads.shape))
print("MHA output:        ", tuple(mha_out.shape))
assert concat_heads.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL)
assert mha_out.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL)


# Cross-check: an independent per-head loop with no reshape tricks at all
# must produce exactly the same answer as your reshape-based version.
def mha_loop(Q, K, V, valid_tokens, W_O):
    outs = []
    for h in range(NUM_HEADS):
        sl = slice(h * D_HEAD, (h + 1) * D_HEAD)
        s = Q[..., sl] @ K[..., sl].transpose(-2, -1) / math.sqrt(D_HEAD)
        a = torch.softmax(ignore_padding(s, valid_tokens), dim=-1)
        outs.append(a @ V[..., sl])
    return W_O(torch.cat(outs, dim=-1))


assert torch.allclose(mha_out, mha_loop(Q, K, V, valid_tokens, W_O), atol=1e-5), \
    "your reshape-based multi-head attention must match an independent per-head loop"
print("matches independent per-head loop: True")


#### What I Did

`combine_heads` puts the 4 heads' 16-dim outputs back together into one `[16, 32, 64]` tensor, and `W_O` is one more learned linear layer that mixes information back across heads after they each attended independently. It matches an independently written per-head loop exactly, confirming the reshape-based version isn't accidentally scrambling anything.

**Analysis Question**

- **Q21.** Why does the representation return to 64 dimensions after the four attention heads are concatenated?

#### My Answers

- **Q21.** Because 4 heads of 16 dimensions each are concatenated back together: `4 x 16 = 64 = d_model`. Splitting never discarded any dimensions, it only grouped the existing 64 columns into 4 chunks of 16, so putting those same 4 chunks back side by side necessarily reconstructs the original width.

### 5.4 Compare Attention Heads

**TODO:** For the same article as in Task 4, plot the attention matrices of **Head 1** and **Head 2** side by side using `plot_attention` (real tokens only).

In [ ]:
plot_attention(mh_weights[EX, 0, :n_show, :n_show].detach(), tokens_ex, "Head 1 (index 0)")
plot_attention(mh_weights[EX, 1, :n_show, :n_show].detach(), tokens_ex, "Head 2 (index 1)")

head_diff = (mh_weights[EX, 0, :n_show, :n_show] - mh_weights[EX, 1, :n_show, :n_show]).abs().sum()
print("Total absolute difference between Head 1 and Head 2:", head_diff.item())


#### What I Did

I plotted the same article's attention matrix for head 0 and head 1 side by side, then summed the absolute difference between the two matrices as one number - a quick way to confirm the heads actually produced different attention patterns, not just visually eyeballing two heatmaps.

**Analysis Questions**

- **Q22.** Are the attention matrices identical?
- **Q23.** Why might different heads learn different relationships?
- **Q24.** Give one example of two different relationships that two attention heads might potentially learn.

#### My Answers

- **Q22.** No, they are not identical - the two heatmaps look visibly different, and the printed total absolute difference is a clearly nonzero number. This makes sense since each head multiplies a different 16-column slice of Q and K (coming from different parts of the same `W_Q`/`W_K` matrices), so even with identical input, each head is comparing different numbers.

- **Q23.** Each head only ever sees its own 16-dimensional slice of Q, K, and V, so it's working from a different, smaller "view" of the same token. Later, once gradients flow during training, each head's slice of `W_Q`/`W_K`/`W_V` can be pushed in a different direction without affecting the other heads, letting different heads become useful for different things instead of all competing to encode everything in one shared score.

- **Q24.** One head might end up favoring short-range relationships, like a token attending strongly to its immediate neighbors (useful for things like adjective-noun pairs), while another head might favor long-range relationships, like a verb attending back to its subject several words earlier in the sentence.

---
## Task 6 — Residual Connections, Layer Normalization and Feed-Forward Network

### 6.1 Residual Connection and Layer Normalization

$$X_1 = \text{LayerNorm}\big(X + \text{MHA}(X)\big)$$

**TODO:** Create `norm1 = nn.LayerNorm(D_MODEL)`. Compute the residual sum and then `X1`.

In [ ]:
norm1 = nn.LayerNorm(D_MODEL)
residual1 = X + mha_out
X1 = norm1(residual1)


# --- test your implementation ---
for name, t in [("X", X), ("MHA output", mha_out), ("After residual", residual1), ("After LayerNorm", X1)]:
    print(f"{name:16s}: {tuple(t.shape)}")
    assert t.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL)
assert torch.allclose(X1.mean(dim=-1), torch.zeros(BATCH_SIZE, MAX_LENGTH), atol=1e-4), \
    "after LayerNorm each token vector should have mean ~0"
assert torch.allclose(X1.var(dim=-1, unbiased=False), torch.ones(BATCH_SIZE, MAX_LENGTH), atol=1e-2), \
    "after LayerNorm each token vector should have variance ~1"


#### What I Did

`residual1 = X + mha_out` adds the attention output back to its own input, and `norm1` renormalizes every token's 64-dim vector to mean 0 and variance 1. The residual add only type-checks because `mha_out` came out of `W_O` with the exact same `[16, 32, 64]` shape as `X`.

**Analysis Question**

- **Q25.** Why must the MHA output have the same shape as X for this residual connection?

#### My Answers

- **Q25.** A residual connection is computed as `X + MHA(X)`, which is an element-wise addition - addition is only defined between tensors of the same shape. If MHA's output had a different shape than `X`, there would be no well-defined way to add them together position by position, so the whole "keep the original signal and add a correction to it" idea only works because every sub-layer in the encoder preserves `[B, L, D]`.

### 6.2 Implement the Feed-Forward Network

The FFN operates **independently on every token** representation:

```text
[16, 32, 64]  --Linear-->  [16, 32, 256]  --ReLU-->  --Linear-->  [16, 32, 64]
```

**TODO:** Build `ffn` as an `nn.Sequential` of `Linear(64, 256)`, `ReLU()` and `Linear(256, 64)`. Apply it to `X1`.

In [ ]:
torch.manual_seed(SEED)

ffn = nn.Sequential(
    nn.Linear(D_MODEL, D_FF),
    nn.ReLU(),
    nn.Linear(D_FF, D_MODEL),
)
ffn_out = ffn(X1)


# --- test your implementation ---
ffn_hidden = ffn[1](ffn[0](X1))
print("FFN input: ", tuple(X1.shape))
print("FFN hidden:", tuple(ffn_hidden.shape))
print("FFN output:", tuple(ffn_out.shape))
assert ffn_hidden.shape == (BATCH_SIZE, MAX_LENGTH, D_FF)
assert ffn_out.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL)
# position-wise: running the FFN on one token alone gives the same answer
assert torch.allclose(ffn(X1[0, 3]), ffn_out[0, 3], atol=1e-5)


#### What I Did

The FFN is just `Linear(64, 256) -> ReLU -> Linear(256, 64)`, applied the same way to every token independently - there's no mixing across positions here at all, which the last assertion confirms directly: running the FFN on one token's vector alone gives the exact same result as running it on the whole batch and reading off that one token.

### 6.3 Add the Second Residual Connection

$$\text{EncoderOutput} = \text{LayerNorm}\big(X_1 + \text{FFN}(X_1)\big)$$

**TODO:** Use a **second** LayerNorm, `norm2`.

In [ ]:
norm2 = nn.LayerNorm(D_MODEL)
encoder_output = norm2(X1 + ffn_out)


# --- test your implementation ---
print("Encoder output:", tuple(encoder_output.shape))
assert norm2 is not norm1, "use a second, separate LayerNorm"
assert encoder_output.shape == (BATCH_SIZE, MAX_LENGTH, D_MODEL)
assert torch.isfinite(encoder_output).all()


#### What I Did

Same pattern as the first residual: add the FFN's output back to its own input (`X1 + ffn_out`), then normalize with a *second*, separately initialized LayerNorm, `norm2`. This finishes one full encoder block and keeps the familiar `[16, 32, 64]` shape all the way through.

**Analysis Questions**

- **Q26.** Self-attention mixes information across tokens. What does the feed-forward network add to the Transformer?
- **Q27.** Why does the FFN first increase the dimension from 64 to 256 and then reduce it back to 64?

#### My Answers

- **Q26.** Self-attention's job is purely to *mix* information across different token positions - it decides how much of token B's content should flow into token A's representation. The feed-forward network does the opposite kind of work: it doesn't look at any other token at all, it just applies a learned non-linear transformation to each token's own (already-mixed) representation independently, giving the model extra capacity to process and reshape what attention gathered, rather than gathering more information itself.

- **Q27.** Expanding from 64 to 256 gives the network a much wider intermediate space to compute a richer non-linear transformation in - with only 64 dimensions throughout, the ReLU non-linearity would have far less room to combine features in useful ways. Projecting back down to 64 afterward is required to match `d_model`, since the next layer (and the residual connection around this one) both expect the standard `[B, L, 64]` shape.

---
## Task 7 — Build and Verify Your Encoder

### 7.1 Build Your Own Encoder Class

**TODO:** Package everything from Tasks 3–6 into one reusable `nn.Module`. It must contain:

- Q, K and V projections (no bias)
- Head splitting
- Scaled dot-product attention
- Instructor-provided padding handling (`ignore_padding`) before softmax
- Head concatenation
- Output projection `W_O`
- Residual connection → LayerNorm
- Feed-forward network
- Second residual connection → second LayerNorm

`forward(x, valid_tokens, return_attention=False)` takes the embedding-plus-position tensor and its `valid_tokens` indicator. It returns the encoder output, plus the attention probabilities `[B, H, L, L]` when `return_attention=True`.

Do **not** use any built-in Transformer or multi-head attention layer. Read `B` and `L` from the input, because the final batch may have fewer than 16 examples.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF, dropout=0.1):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be evenly divisible by num_heads ({num_heads}).")
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, L, D = x.shape
        x = x.reshape(B, L, self.num_heads, self.d_head)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        B, H, L, d_head = x.shape
        x = x.transpose(1, 2)
        return x.reshape(B, L, H * d_head)

    def forward(self, x, valid_tokens, return_attention=False):
        Q = self.split_heads(self.W_Q(x))
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        scores = ignore_padding(scores, valid_tokens)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        context = torch.matmul(attn, V)
        context = self.combine_heads(context)
        attn_out = self.W_O(context)

        x1 = self.norm1(x + attn_out)
        out = self.norm2(x1 + self.ffn(x1))

        if return_attention:
            return out, attn
        return out


#### What I Did

This class is just everything from Tasks 3-6 packaged into one reusable module: project to Q/K/V, split into heads, scaled dot-product attention with padding masked out, dropout on the attention probabilities, combine the heads back together, `W_O`, then the two residual-plus-LayerNorm blocks around attention and the feed-forward network. `forward` reads `B` and `L` from the input rather than assuming `BATCH_SIZE`, so it works for the final, smaller batch too.

### 7.2 Verify Your Encoder

Run the cell below as given. Every check must pass. The encoder is put in `eval()` mode so dropout is switched off and the attention rows sum exactly to one.

In [ ]:
torch.manual_seed(SEED)
encoder = TransformerEncoderBlock()
encoder.eval()                       # disables dropout for these checks

with torch.no_grad():
    enc_out, enc_attn = encoder(X, valid_tokens, return_attention=True)

B = X.shape[0]
print("Input shape:          ", tuple(X.shape))
print("Encoder output shape: ", tuple(enc_out.shape))
print("Attention probs shape:", tuple(enc_attn.shape))

assert X.shape == (B, MAX_LENGTH, D_MODEL)
assert enc_out.shape == (B, MAX_LENGTH, D_MODEL)
assert torch.isfinite(enc_out).all(), "encoder output contains NaN or inf"
assert enc_attn.shape == (B, NUM_HEADS, MAX_LENGTH, MAX_LENGTH)
assert torch.allclose(enc_attn.sum(dim=-1), torch.ones(B, NUM_HEADS, MAX_LENGTH), atol=1e-5), \
    "each attention row must sum to ~1"
assert (enc_attn * (~valid_tokens)[:, None, None, :]).abs().max().item() < 1e-6, \
    "padding key positions must receive ~0 attention"

# A smaller (final) batch must also work
with torch.no_grad():
    small_out = encoder(X[:5], valid_tokens[:5])
assert small_out.shape == (5, MAX_LENGTH, D_MODEL)

# Without return_attention, only the output is returned
assert torch.is_tensor(small_out)

# No built-in attention / Transformer layers
banned = (nn.MultiheadAttention, nn.Transformer, nn.TransformerEncoder, nn.TransformerEncoderLayer)
assert not any(isinstance(m, banned) for m in encoder.modules()), "built-in attention layers are not allowed"

print("\nAll encoder checks passed.")
print("Trainable parameters:", sum(p.numel() for p in encoder.parameters()))

In [ ]:
n_show7 = min(15, int(valid_tokens[EX].sum()))
tokens_ex7 = tokenizer.encode(train_texts[EX]).tokens[:n_show7]

plot_attention(enc_attn[EX, 0, :n_show7, :n_show7].detach(), tokens_ex7, "Encoder class - Head 0")
plot_attention(enc_attn[EX, 3, :n_show7, :n_show7].detach(), tokens_ex7, "Encoder class - Head 3")


#### What I Did

Same `plot_attention` helper as before, now pointed at the attention probabilities returned by the packaged `TransformerEncoderBlock` itself, to confirm the class's internal attention behaves the same way as the manual, step-by-step version from Tasks 3-5.

---
## Lab 5 Shape Trace

Fill in the **Your Shape** column with the shapes printed by your code.

| Stage | Expected Shape | Your Shape |
|---|---|---|
| Token IDs | `[B, 32]` | `[16, 32]` |
| Valid-token indicator | `[B, 32]` | `[16, 32]` |
| Token embeddings | `[B, 32, 64]` | `[16, 32, 64]` |
| Positional encoding | `[32, 64]` | `[32, 64]` |
| Embedding + position | `[B, 32, 64]` | `[16, 32, 64]` |
| Q, K, V before splitting | `[B, 32, 64]` each | `[16, 32, 64]` each |
| Q, K, V after splitting | `[B, 4, 32, 16]` each | `[16, 4, 32, 16]` each |
| Attention scores / probabilities | `[B, 4, 32, 32]` | `[16, 4, 32, 32]` |
| Attention output per head | `[B, 4, 32, 16]` | `[16, 4, 32, 16]` |
| Concatenated heads | `[B, 32, 64]` | `[16, 32, 64]` |
| MHA output | `[B, 32, 64]` | `[16, 32, 64]` |
| First residual + LayerNorm | `[B, 32, 64]` | `[16, 32, 64]` |
| FFN hidden representation | `[B, 32, 256]` | `[16, 32, 256]` |
| Encoder output | `[B, 32, 64]` | `[16, 32, 64]` |

(`B = 16` for every batch used in Tasks 2-7, since `token_ids = train_ids[:BATCH_SIZE]`; Task 7.2 additionally confirms the same class also works correctly for a smaller final batch of `B = 5`.)

---
## Deliverable

Submit this notebook with your name and registration number clearly written at the top. It must include:

- all `TODO` cells completed, with their test/validation cells executed successfully;
- the BPE tokenization examples and at least one subword-splitting example;
- the single-head and multi-head attention heatmaps from the untrained encoder;
- the completed shape trace and the passing encoder checks from Task 7.2;
- concise written answers to **Q1–Q27**.